In [20]:
# ================== Install dependencies in Colab ==================
!pip install -q langchain langchain-openai langchain-groq langchain-community faiss-cpu pandas openpyxl sentence-transformers
#!pip install -q langchain langchain_openai faiss-cpu openpyxl pandas langchain-groq
#!pip install -q langchain langchain-community langchain-groq


import pandas as pd
import os
#from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS   # ✅ updated import

# import pandas as pd
# from langchain_openai import OpenAIEmbeddings, ChatOpenAI
# from langchain_groq import ChatGroq
# from langchain.prompts import ChatPromptTemplate
# from langchain.vectorstores import FAISS
# import os

# ================== STEP 1: Load dataset ==================
products = pd.read_excel("/content/Product_Email_Dataset.xlsx", sheet_name="Products")
emails = pd.read_excel("/content/Product_Email_Dataset.xlsx", sheet_name="Client Emails")

print("Products:")
print(products.head())

# ================== STEP 2: Create Vectorstore of product descriptions ==================
#embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Convert products into documents for retrieval
product_texts = [
    f"{row['Description']} | Session: {row['Seasons']} | Price: {row['Price']} | Stock: {row['Stock']}"
    for _, row in products.iterrows()
]

vectorstore = FAISS.from_texts(product_texts, embeddings)

# ================== STEP 3: LLM initialization ==================

os.environ["GROQ_API_KEY"] = "YOUR_API_KEY_HERE"
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
#llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

prompt_template = ChatPromptTemplate.from_template(
    """
    You are a helpful customer support assistant.

    Customer email:
    {email_text}

    Retrieved products from database:
    {retrieved_products}

    Instructions:
    - If the email asks about a product, provide details (price & stock) from the retrieved products.
    - If product is not available, say it is out of stock.
    - If the email is unrelated to products, politely give a generic response.
    """
)

# ================== STEP 4: Response generation with retrieval ==================
def generate_response(email_subject, email_body, top_k=3):
    email_text = (email_subject or "") + " " + (email_body or "")

    # Retrieve most relevant products
    docs = vectorstore.similarity_search(email_text, k=top_k)
    retrieved_products = "\n".join([doc.page_content for doc in docs])

    chain = prompt_template | llm
    response = chain.invoke({"email_text": email_text, "retrieved_products": retrieved_products})
    return response.content

# ================== STEP 5: Test ==================
sample = emails.sample(5)
for _, row in sample.iterrows():
    print("\nEmail:", row["email_subject"], row["email_body"])
    print("Response:", generate_response(row["email_subject"], row["email_body"]))


Products:
  Product ID                 Name         Category  \
0       P001       Smart Widget X    Smart Devices   
1       P002     Smart Thermostat  Home Automation   
2       P003      Pro Gadget Pack    Smart Devices   
3       P004  Eco-Friendly Widget    Smart Devices   
4       P005     Wireless Speaker            Audio   

                                         Description  Stock        Seasons  \
0  Compact smart widget with voice control and Io...    150    All Seasons   
1  Energy-saving thermostat with app control and ...     80    All Seasons   
2  Bundle of smart devices including a hub plug a...     50    All Seasons   
3  Sustainable widget made from recycled material...    200  Spring/Summer   
4  Portable Bluetooth speaker with high-fidelity ...    120    All Seasons   

    Price  
0   59.99  
1  129.99  
2  199.99  
3   39.99  
4   79.99  


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Email: Need info on Custom Gadget Hello, I am interested in Custom Gadget. Could you let me know if it's available and the price?
Response: Subject: Re: Custom Gadget Inquiry

Dear [Customer],

Thank you for your interest in our Custom Gadget. We have a few options available that might interest you.

We have three Custom Gadget options:

1. Entry-level smart widget with basic automation features and sleek design. This gadget is available in all seasons and is priced at $29.99. We currently have 180 units in stock.
2. Customizable gadget with color options and modular add-ons for personalization. This gadget is also available in all seasons and is priced at $69.99. Unfortunately, we are currently out of stock with only 100 units available.
3. Ultra-compact gadget for basic smart home control perfect for small spaces. This gadget is available in all seasons and is priced at $19.99. We currently have 175 units in stock.

Please let me know if you have any further questions or if there's 